In [133]:
!pip install fastf1

In [134]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [135]:
print("\n🏎️  F1 Lap Time Predictor")
print("="*40)

#user input
season = int(input("Enter season (e.g., 2023, 2024): "))
round_num = input("Enter round number (or press Enter for latest): ")

#if user input for round num is empty
if round_num.strip()=="":
  schedule = fastf1.get_event_schedule(season)
  round_num = len(schedule)
  print(f"Latest race: Round {round_num}")
else:
  round_num = int(round_num)

#loading the session (R=race)
print(f"\n📊 Loading season {season}, round {round_num}...")
session = fastf1.get_session(season, round_num,"R")
session.load(laps=True)



🏎️  F1 Lap Time Predictor
Enter season (e.g., 2023, 2024): 2026
Enter round number (or press Enter for latest): 1

📊 Loading season 2026, round 1...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core   

In [136]:
#data preprocessing

print("\n🧹 Cleaning data...")
laps = session.laps.copy() #get laps data

laps = laps.dropna(subset=["LapTime"])

laps = laps[laps["LapNumber"]>1] #remove lap 1

if "PitInTime" in laps.columns:
  laps = laps[laps["PitInTime"].isna()] #remove pit laps

laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds() #convert time to secs

#remove outliers using IQR method
Q1 = laps["LapTimeSeconds"].quantile(0.25)
Q3 = laps["LapTimeSeconds"].quantile(0.75)
IQR = Q3-Q1
lower = Q1-1.5*IQR
lower = Q3+1.5*IQR
laps = laps[(laps['LapTimeSeconds'] >= lower) & (laps['LapTimeSeconds'] <= upper)]

print(f"✅ Cleaned data: {len(laps)} laps remaining")
laps.head()


🧹 Cleaning data...
✅ Cleaned data: 95 laps remaining


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,LapTimeSeconds
11,0 days 01:20:29.787000,NOR,1,0 days 00:01:58.075000,12.0,2.0,0 days 01:18:34.141000,NaT,0 days 00:00:44.892000,0 days 00:00:27.737000,...,McLaren,0 days 01:18:31.712000,2026-03-08 04:19:38.335,6,9.0,False,,False,False,118.075
12,0 days 01:22:21.719000,NOR,1,0 days 00:01:51.932000,13.0,2.0,NaT,NaT,0 days 00:00:41.529000,0 days 00:00:27.589000,...,McLaren,0 days 01:20:29.787000,2026-03-08 04:21:36.410,671,9.0,False,,False,False,111.932
17,0 days 01:29:33.105000,NOR,1,0 days 00:01:35.404000,18.0,2.0,NaT,NaT,0 days 00:00:29.758000,0 days 00:00:17.788000,...,McLaren,0 days 01:27:57.701000,2026-03-08 04:29:04.324,126,5.0,False,,False,False,95.404
18,0 days 01:31:29.175000,NOR,1,0 days 00:01:56.070000,19.0,2.0,NaT,NaT,0 days 00:00:42.456000,0 days 00:00:22.738000,...,McLaren,0 days 01:29:33.105000,2026-03-08 04:30:39.728,67,5.0,False,,False,False,116.070
69,0 days 01:20:35.580000,GAS,10,0 days 00:01:58.072000,12.0,2.0,0 days 01:18:39.918000,NaT,0 days 00:00:42.291000,0 days 00:00:29.532000,...,Alpine,0 days 01:18:37.508000,2026-03-08 04:19:44.131,6,11.0,False,,False,False,118.072


In [137]:
#features

print("\n🔧 Creating features...")

features = pd.DataFrame()
features["LapNumber"] = laps["LapNumber"]
features["TyreLife"] = laps["TyreLife"].fillna(0) #0 for missing values

if "Stint" in laps.columns:
  features["Stint"] = laps["Stint"].fillna(1)

if "Position" in laps.columns:
    features["Position"] = laps["Position"].fillna(20)

#encoding driver names to nums
driver_ids = {driver: i for i, driver in enumerate(laps["Driver"].unique())}
features["DriverCode"] = laps["Driver"].map(driver_ids)

#tyre compounds to nums
if "Compound" in laps.columns:
    compound_map = {"SOFT": 0, "MEDIUM": 1, "HARD": 2, "INTERMEDIATE": 3, "WET": 4}
    features["Compound"] = laps["Compound"].map(compound_map).fillna(0)

target = laps["LapTimeSeconds"] #target variable
print(f"✅ Created {features.shape[1]} features")


🔧 Creating features...
✅ Created 6 features


In [139]:
#train model
print("\n🤖 Training model...")

#split data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 42)

#random forest
model = RandomForestRegressor(n_estimators = 100, max_depth = 10, random_state = 42)
model.fit(X_train, y_train)

#evaluate
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"\n📈 Model Performance:")
print(f"   Mean Absolute Error: {mae:.3f} seconds")
print(f"   R² Score: {r2:.3f}")


🤖 Training model...

📈 Model Performance:
   Mean Absolute Error: 3.153 seconds
   R² Score: 0.771


In [140]:
print("\n🔍 Most important factors:")
importance_df = pd.DataFrame({
    "Feature": features.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

for i, row in importance_df.head(5).iterrows():
    print(f"   {row["Feature"]}: {row["Importance"]:.2%}")


🔍 Most important factors:
   LapNumber: 56.37%
   TyreLife: 18.94%
   Position: 16.29%
   DriverCode: 5.41%
   Stint: 1.70%


In [142]:
#predictions
print("\n" + "="*40)
print("🎯 Make Your Own Prediction")
print("="*40)

#user input
print("\nEnter race conditions:")
lap = int(input("  Lap number (e.g., 20): "))
tyre_life = int(input("  Tyre age in laps (e.g., 8): "))
stint = int(input("  Stint number (e.g., 2): "))
position = int(input("  Race position (e.g., 5): "))
driver_num = driver_ids.get(input("  Driver code (e.g., VER, HAM, LEC): ").upper(),0) #converting to num
compound_num = compound_map.get(input("  Tire compound (SOFT/MEDIUM/HARD): ").upper(),0) #converting to num

#prediction input
pred_data = pd.DataFrame([[lap, tyre_life, stint, position, driver_num, compound_num]], columns=features.columns)

#make predictions
pred_time = model.predict(pred_data)[0]

print(f"\n⏱️  Predicted lap time: {pred_time:.3f} seconds")
print(f"   (That's {pred_time/60:.3f} minutes if you're curious)")


🎯 Make Your Own Prediction

Enter race conditions:
  Lap number (e.g., 20): 20
  Tyre age in laps (e.g., 8): 6
  Stint number (e.g., 2): 2
  Race position (e.g., 5): 4
  Driver code (e.g., VER, HAM, LEC): ham
  Tire compound (SOFT/MEDIUM/HARD): soft

⏱️  Predicted lap time: 101.259 seconds
   (That's 1.688 minutes if you're curious)


In [143]:
import joblib

# Save model and features
joblib.dump(model, 'f1_model.pkl')
joblib.dump(features.columns.tolist(), 'feature_columns.pkl')

# Download to your computer
from google.colab import files
files.download('f1_model.pkl')
files.download('feature_columns.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>